In [1]:
!pip install 'powerlaw'

In [2]:
import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe

b2.get_file('modules/metaorder.py')
b2.get_file('modules/LMF.py')

import metaorder as metaorder
import LMF as LMF

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp 
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from scipy.stats import t
from scipy.special import gamma as gamma_func

In [3]:
trader_participation_distribution = 'power'
group                             = 'year'
config                            = 'metaorder'

In [4]:

if trader_participation_distribution == 'power':
    param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500], 'delta': [1.5, 2, 3, 4, 5]}
if trader_participation_distribution == 'homogenous':
    param_grid = {'N': [5, 10, 20, 30, 40, 50, 100, 500, 1000, 1500]}

expanded_grid = list(ParameterGrid(param_grid))
print(expanded_grid)

[{'N': 5, 'delta': 1.5}, {'N': 5, 'delta': 2}, {'N': 5, 'delta': 3}, {'N': 5, 'delta': 4}, {'N': 5, 'delta': 5}, {'N': 10, 'delta': 1.5}, {'N': 10, 'delta': 2}, {'N': 10, 'delta': 3}, {'N': 10, 'delta': 4}, {'N': 10, 'delta': 5}, {'N': 20, 'delta': 1.5}, {'N': 20, 'delta': 2}, {'N': 20, 'delta': 3}, {'N': 20, 'delta': 4}, {'N': 20, 'delta': 5}, {'N': 30, 'delta': 1.5}, {'N': 30, 'delta': 2}, {'N': 30, 'delta': 3}, {'N': 30, 'delta': 4}, {'N': 30, 'delta': 5}, {'N': 40, 'delta': 1.5}, {'N': 40, 'delta': 2}, {'N': 40, 'delta': 3}, {'N': 40, 'delta': 4}, {'N': 40, 'delta': 5}, {'N': 50, 'delta': 1.5}, {'N': 50, 'delta': 2}, {'N': 50, 'delta': 3}, {'N': 50, 'delta': 4}, {'N': 50, 'delta': 5}, {'N': 100, 'delta': 1.5}, {'N': 100, 'delta': 2}, {'N': 100, 'delta': 3}, {'N': 100, 'delta': 4}, {'N': 100, 'delta': 5}, {'N': 500, 'delta': 1.5}, {'N': 500, 'delta': 2}, {'N': 500, 'delta': 3}, {'N': 500, 'delta': 4}, {'N': 500, 'delta': 5}, {'N': 1000, 'delta': 1.5}, {'N': 1000, 'delta': 2}, {'N': 

In [5]:
b2.get_file('top_250.csv')
top_250 = pd.read_csv('top_250.csv')

top_250_tickers = top_250['Ticker'].tolist()

In [6]:
%%time
files = b2.list_files(path = 'data/trades/top_250')

for f in files:
    b2.get_file(f'data/trades/top_250/{f}')

CPU times: user 10.8 s, sys: 5.36 s, total: 16.2 s
Wall time: 1min 11s


In [7]:
%%time
files = b2.list_files(path = 'data/trades/top_250')

for f in files:
    b2.get_file(f'data/trades/top_250/{f}')

def load_stock(parquet):
    try:
        df = pl.read_parquet(parquet)
        return df.sort(['DateTime', 'ExchangeSequenceNo'])
    except FileNotFoundError:
        return None

exchange = 'XJSE'
stocks = {
    ticker: df
    for ticker in top_250_tickers
    if (df := load_stock(f'{ticker}_{exchange}.parquet')) is not None
}

CPU times: user 1min 3s, sys: 24.8 s, total: 1min 28s
Wall time: 1min 7s


In [8]:
%%time
metaorder.metaorder_stylised_facts(stocks['GRT'], N = 100)

/home/bmll/local/metaorder.py:559: OptimizeWarning:

Covariance of the parameters could not be estimated



CPU times: user 1min 13s, sys: 17 s, total: 1min 30s
Wall time: 1min 1s


year,ticker,N,N (st),delta,xmin,alpha,intercept (nlls),gamma (nlls),N (nlls),intercept (psd),gamma (psd),N (psd),impact curve exponent,impact curve exponent var,impact curve half width,phi exponent,phi exponent var,phi exponent half width,beta,beta var,beta half width
i64,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023,"""GRT""",100,52,2.0,3.0,1.905225,0.734663,0.500202,0.82317,0.376409,0.482646,3.08753,0.074267,0.000757,0.055733,0.869865,0.001025,0.064369,4.1963e-9,inf,inf
2024,"""GRT""",100,76,2.0,3.0,1.837256,0.375304,0.340167,2.833499,0.440042,0.433737,2.255569,29.685004,1.2975e9,72984.261235,0.739666,0.001375,0.074551,0.000004,1.0817e-12,0.000002
2025,"""GRT""",100,48,2.0,3.0,1.919214,0.373733,0.393937,2.932723,0.318142,0.489648,4.319512,0.127933,0.000832,0.058452,0.723623,0.001234,0.070629,0.000001,1.4228e-13,7.5841e-7


In [ ]:
trader_participation_distribution = 'power'

b2.create_folder('data', ensure_exists = True)
b2.create_folder('data/grid_search_7', ensure_exists = True)
b2.create_folder(f'data/grid_search_7/{exchange}', ensure_exists = True)
b2.create_folder(f'data/grid_search_7/{exchange}/{trader_participation_distribution}', ensure_exists = True)

In [ ]:
%time

start_from = 'BHG'
start = False

counter = 1
for ticker, stock_data in stocks.items():

    if ticker == start_from:
        start = True

    if not start:
        continue
        
    print(f'{counter}: {ticker}')
    counter += 1

    results_list = []
    for params in expanded_grid:
        
        N        = params['N']
        alpha_tp = params['alpha_tp']
        print(f'{N} {alpha_tp}')
    
        res = metaorder.metaorders_stylised_facts(stock_data, N, alpha_tp = alpha_tp,
                                        trader_participation_distribution = trader_participation_distribution)
    
        if not res.is_empty():
            results_list.append(res)

    if results_list:
        results_df = pl.concat(results_list, how = 'vertical', rechunk = True)
        results_df.write_parquet(f'{ticker}_{exchange}_{trader_participation_distribution}_grid_search.parquet')
        b2.put_file(f'{ticker}_{exchange}_{trader_participation_distribution}_grid_search.parquet', f'data/grid_search_7/{exchange}/{trader_participation_distribution}')
    else:
        results_df = pl.DataFrame()
        